# Later methods for the Drugs.com reviews

The course notebook (`../project-code.ipynb`) already compared TF-IDF logistic regression, LinearSVC, XGBoost, histogram gradient boosting, a Bi-LSTM, and a frozen DistilBERT embedding. Those runs re-split the official training file and used only the review text.

This notebook trains the methods that were still open:

- Complement Naive Bayes on word TF-IDF and on character n-grams
- Logistic regression with `drugName`, `condition`, `usefulCount`, and review length
- Frank–Hall ordinal logistic regression (Negative < Neutral < Positive)
- LightGBM on a 120-d LSA vector plus the same metadata
- A star regressor that predicts the 1–10 rating and then applies the 1–4 / 5–6 / 7–10 bins
- LSA plus effectiveness / side-effect sentence features (VADER). The file has no aspect labels
- A word-level text CNN
- DistilBERT fine-tuned end to end on a stratified sample

Train is `dataset/drugsComTrain_raw.csv`. Test is `dataset/drugsComTest_raw.csv`. Read **macro-F1** next to accuracy. The DistilBERT row uses a 6,000 / 3,000 stratified sample, so it is not on the full test file.

Kernel: `drug-review`. Create it with `uv sync` in this directory.

In [ ]:
import pandas as pd
from drug_review.data import RESULTS

# Retrain with: from drug_review.pipeline import run; run()
metrics = pd.read_csv(RESULTS / "metrics.csv")
metrics.round(4)

## What the official test file showed

111,070 cleaned training rows, 47,696 cleaned test rows. Positive is 66% of train, Neutral is 9%.

**Word TF-IDF logistic regression, no class weights, is the leader on both accuracy (0.857) and macro-F1 (0.688).** Neutral F1 is still 0.332. This is the same model family as the course baseline. The course accuracy of 0.796 is a different test set: a re-split of the training file, not this UCI test file.

**The useful gain is on Neutral, and it comes from class weights plus the columns the course notebook ignored.** Balanced logistic regression on TF-IDF plus drug, condition, usefulCount, and review length reaches Neutral F1 0.418 (accuracy 0.774). LightGBM on a 120-d LSA vector plus those columns has the best Neutral F1, **0.453**, at accuracy 0.757.

The other ideas did not beat that:

- Frank–Hall ordinal logistic keeps accuracy at 0.822 and drops Neutral F1 to 0.079. Forcing the order Negative < Neutral < Positive did not recover the thin middle.
- Predicting the 1–10 star and then binning is weaker than a 3-class model on the same vectors (macro-F1 0.505 vs 0.555). Errors around stars 4–7 land in the wrong bin.
- Aspect cues fire on 75% of training reviews and lift LSA logistic regression only slightly (macro-F1 0.567 vs 0.555). The file has no aspect labels, so this is a lexicon feature, not annotated aspect classification.
- Character n-grams are the weakest row (macro-F1 0.502).
- The text CNN (3 epochs) reaches Neutral F1 0.299 and macro-F1 0.572, behind the linear models.
- Fine-tuned DistilBERT, one epoch on 6,000 stratified reviews, scores 0.745 accuracy and Neutral F1 0.209 on a 2,999-row stratified slice. That is a real fine-tune, not a frozen embedding, and it does not beat TF-IDF at this sample size. The training code is in `drug_review/models.py` if you run it on the full split with a GPU.

Per-condition scores below are for the macro-F1 leader (word TF-IDF logistic regression). Birth Control is easier (macro-F1 0.748). Pain is harder (0.602) even though its accuracy is high, because accuracy there is mostly the Positive class.


In [ ]:
from IPython.display import Image, display
import pandas as pd
from drug_review.data import RESULTS

display(Image(filename=str(RESULTS / "comparison.png")))
pd.read_csv(RESULTS / "per_condition.csv").round(4)